# Data Types

GDAL stores every raster band in one fixed C type, such as `Byte` (8-bit unsigned) or `Float32`, and OGR stores every vector field as a typed column. Many ETHOS.GeoKit functions create rasters or fields for you, so GeoKit has to choose these types. A type that is too narrow clips, wraps around or truncates values without notice; a type that is too wide costs memory and disk space.

This example shows the `dtype` parameter that every function writing a raster shares, its three automatic modes, explicit types, the helpers of `geokit.dtypes`, and how field types follow from column types. The reasons behind these rules are recorded in the [Data Types](../../explanation/data_types/index.html) section of the documentation.

In [ ]:
import warnings

import numpy as np
import pandas as pd
from osgeo import gdal

import geokit as gk
from geokit import dtypes
from geokit.error import GeoKitDataTypeError


def band_type(raster):
    """The GDAL name of the data type of the first band."""
    return gk.raster.rasterInfo(raster).data_type_name_str


grid = dict(bounds=(0, 0, 1000, 1000), pixelWidth=100, pixelHeight=100, srs=3035)

## The default: `dtype="auto"`

Without a `dtype`, GeoKit chooses a type that holds every value the operation can produce. The choice follows from the input types and the operation, never from the data, so the same call gives the same type for every input of the same type. A raster created from an array keeps the array's type, a raster created from nothing is `Byte`, and a constant burn value gets the smallest type that holds it: `Byte` comes before `Int8`, and from 16 bits on the signed type comes before the unsigned one.

In [ ]:
small_values = (np.arange(100).reshape(10, 10) % 6).astype(np.int64)
print("createRaster(data=int64 array) ->", band_type(gk.raster.createRaster(data=small_values, **grid)))
print("createRaster() without data    ->", band_type(gk.raster.createRaster(**grid)))

square = gk.geom.box(0, 0, 500, 500, srs=3035)
for burn_value in [1, 200, 300, 40000, -1, 0.1]:
    burned = gk.vector.rasterize(square, 100, 100, value=burn_value)
    print(f"rasterize(value={burn_value!r:>6}) -> {band_type(burned)}")

### Operations that widen the values

Averaging or interpolating an integer raster gives fractions, and a sum can exceed the input type. Under `"auto"` the type widens with the operation, so nothing is rounded or clipped: the interpolating resampling algorithms give `Float32` (or `Float64` for 32-bit and wider integers), `sum` gives `Float64`. Without `resampleAlg`, `warp` uses `"auto"`: nearest neighbour for integer rasters, which keeps their values and their type, and bilinear for float rasters. To interpolate an integer raster, such as an elevation model, pass an interpolating algorithm such as `resampleAlg="bilinear"`.

In [ ]:
mask_values = np.zeros((10, 10), np.uint8)
mask_values[:, :3] = 1
byte_mask = gk.raster.createRaster(data=mask_values, **grid)

averaged = gk.raster.warp(byte_mask, resampleAlg="average", pixelWidth=500, pixelHeight=500)
print("warp(average) of a Byte mask ->", band_type(averaged), gk.raster.extractMatrix(averaged)[0])

summed = gk.raster.warp(byte_mask, resampleAlg="sum", pixelWidth=500, pixelHeight=500)
print("warp(sum) of a Byte mask     ->", band_type(summed), gk.raster.extractMatrix(summed)[0])

## `dtype="preserve_input"`: the GDAL convention

The output keeps the type of the input, as GDAL, rasterio and QGIS do. GeoKit does not check the values: fractional results are rounded and sums are clipped by GDAL, without a warning. Use it for categorical data that must stay in its type, or when memory matters more than the exact fractions.

In [ ]:
kept = gk.raster.warp(byte_mask, resampleAlg="average", pixelWidth=500, pixelHeight=500, dtype="preserve_input")
print("preserve_input ->", band_type(kept), gk.raster.extractMatrix(kept)[0])

## `dtype="smallest"`: the narrowest lossless type

GeoKit chooses the type as under `"auto"`, runs the operation, reads the finished output once and shrinks it to the smallest type that stores every value and the noData value exactly. This is the only mode whose type depends on the data, and the read is the price of it.

In [ ]:
shrunk = gk.raster.createRaster(data=small_values, dtype="smallest", **grid)
print('createRaster(data=int64 holding 0..5, dtype="smallest") ->', band_type(shrunk))

shrunk_sum = gk.raster.warp(byte_mask, resampleAlg="sum", pixelWidth=500, pixelHeight=500, dtype="smallest")
print('warp(sum, dtype="smallest")                             ->', band_type(shrunk_sum))

## Explicit types

An explicit `dtype` is used exactly as given, in any spelling: GDAL names with or without the `GDT_` prefix, NumPy names and types, or the Python types `bool`, `int` and `float`. GeoKit does not check the values against it, so the same losses as under `"preserve_input"` can occur. It raises a `GeoKitDataTypeError` only for a request it cannot carry out: a noData, fill or burn value that the type cannot store, or a bare integer such as `gdal.GDT_Float32`, because GDAL and OGR constants share the same integers.

In [ ]:
for requested in ["UInt16", "GDT_Float32", "uint8", np.float64, bool]:
    print(f"dtype={requested!r:<22} -> {band_type(gk.raster.createRaster(dtype=requested, **grid))}")

try:
    gk.raster.createRaster(dtype="UInt16", noData=-1, **grid)
except GeoKitDataTypeError as error:
    print("\n", error)

try:
    gk.raster.createRaster(dtype=gdal.GDT_Float32, **grid)
except GeoKitDataTypeError as error:
    print("\n", error)

## The helpers of `geokit.dtypes`

Inside GeoKit a type is always a `numpy.dtype`. The same helpers are available to you: `to_dtype` converts any spelling, `dtype_for_value` finds the smallest type that stores a value, and `can_hold` tells whether a type stores a value exactly.

In [ ]:
print(dtypes.to_dtype("Byte"), dtypes.to_dtype("GDT_Int16"), dtypes.to_dtype(bool))
for value in [1, 200, 300, 40000, -1, 0.1, np.nan]:
    print(f"dtype_for_value({value!r:>6}) -> {dtypes.gdal_type_name(dtypes.dtype_for_value(value))}")
print('can_hold("Byte", 256):', dtypes.can_hold("Byte", 256))
print('can_hold("Float32", 2**24 + 1):', dtypes.can_hold("Float32", 2**24 + 1))

## Turning the warnings off

Four situations issue a `GeoKitDataTypeWarning`: 64-bit integer data with a NaN noData value becomes `Float64`, which stores whole numbers exactly only up to 2**53; a warp whose output has pixels outside the source (a reprojection, larger bounds or a cutline) without a noData value, which GDAL fills with 0; `polygonizeRaster` of a float raster, whose values `gdal.Polygonize` rounds to integers; and `extractFeatures` of an `Integer64` field with missing values and a value beyond 2**53, which pandas stores as `float64`. `GeoKitDataTypeWarning` is a `UserWarning`, so Python's warning filters turn it off: `warnings.filterwarnings("ignore", category=GeoKitDataTypeWarning)` for the whole process, or the same call inside `warnings.catch_warnings()` for one block. The types and the errors stay the same.

In [ ]:
big_ids = np.array([[123456789, 987654321]], np.int64)
id_raster = gk.raster.createRaster(data=big_ids, bounds=(0, 0, 200, 100), pixelWidth=100, pixelHeight=100, srs=3035)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warped = gk.raster.warp(id_raster, noData=np.nan, resampleAlg="near")
data_type_warnings = [str(w.message) for w in caught if issubclass(w.category, dtypes.GeoKitDataTypeWarning)]
print(band_type(warped), "-", data_type_warnings)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", category=dtypes.GeoKitDataTypeWarning)
    gk.raster.warp(id_raster, noData=np.nan, resampleAlg="near")
data_type_warnings = [w for w in caught if issubclass(w.category, dtypes.GeoKitDataTypeWarning)]
print("with the warning filtered out:", len(data_type_warnings), "data-type warnings")

## Field types of vectors

The `dtype` modes do not apply to vectors; the OGR field type follows from the column type. Integers up to 32 bits become `Integer`, `uint32` and 64-bit integers become `Integer64`, floats become `Real`, strings become `String`, and missing values are written as NULL. `vectorInfo` reports the field types as `numpy.dtype` in `attribute_dtypes`.

In [ ]:
points = [gk.geom.point(6.1, 50.1, srs=4326), gk.geom.point(6.2, 50.2, srs=4326)]
table = pd.DataFrame(
    {
        "geom": points,
        "count": np.array([1, 3_000_000_000], np.uint32),
        "share": [0.5, np.nan],
        "name": ["a", "b"],
    }
)
vector = gk.vector.createVector(table)

print(gk.vector.vectorInfo(vector).attribute_dtypes)
print(gk.vector.extractFeatures(vector)[["count", "share", "name"]])